# 03 Build Silver Tables

Transform Bronze MMSDM rows into typed Silver tables at stable grains for price, demand, regional dispatch, interconnector flows, and generation where source columns are available.

## Configure Silver Transformation Run

This cell creates a run ID and imports Spark SQL helpers used for typed transformations and windowed calculations.

In [ ]:
# Cell purpose: Configure Silver Transformation Run.
from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window

run_id = str(uuid.uuid4())
print(f"run_id={run_id}")

## Bootstrap Local Project Package

This cell makes the uploaded `nem_fabric` source package importable in Fabric. Upload `src/nem_fabric` to `Files/libs/nem_fabric` before running the notebook in a Pipeline.

In [ ]:
# Cell purpose: Make nem_fabric importable from Lakehouse Files.
import os
import sys

fabric_lib_path = os.getenv("FABRIC_NOTEBOOK_LIB_PATH", "/lakehouse/default/Files/libs")
if fabric_lib_path not in sys.path:
    sys.path.insert(0, fabric_lib_path)

print(f"Python library path ready: {fabric_lib_path}")

## Define Shared Silver Helpers

This cell defines NEM region mapping, table checks, and region-name enrichment. It also confirms that Bronze data exists.

In [ ]:
# Cell purpose: Define Shared Silver Helpers.
REGION_MAP = {
    "QLD1": "Queensland",
    "NSW1": "New South Wales",
    "VIC1": "Victoria",
    "SA1": "South Australia",
    "TAS1": "Tasmania",
}


def table_exists(table_name: str) -> bool:
    """Return True when a Lakehouse table exists in the current Spark catalogue."""
    return spark.catalog.tableExists(table_name)


def with_region_name(df, region_col="region"):
    """Add a business-friendly region name using Spark expressions."""
    mapping_expr = F.create_map([item for pair in REGION_MAP.items() for item in (F.lit(pair[0]), F.lit(pair[1]))])
    return df.withColumn("region_name", mapping_expr[F.col(region_col)])


if not table_exists("nem_bronze_mmsdm_rows"):
    raise RuntimeError("nem_bronze_mmsdm_rows does not exist. Run notebook 02 first.")

bronze = spark.table("nem_bronze_mmsdm_rows")

## Build Silver Price and Demand

This cell joins Dispatch `PRICE` and `REGIONSUM` rows into the main 5-minute regional grain and writes the core Silver tables.

In [ ]:
# PRICE and REGIONSUM rows share interval and region keys. Joining them creates the main 5-minute price/demand grain.
price = (
    bronze.filter((F.col("package_name") == "DISPATCH") & (F.col("table_name") == "PRICE"))
    .select(
        F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias("settlement_datetime"),
        F.upper(F.col("regionid")).alias("region"),
        F.col("intervention").cast("int").alias("intervention"),
        F.col("rrp").cast("double").alias("price_aud_mwh"),
        F.col("source_url"),
        F.col("source_zip_name"),
        F.col("row_hash").alias("price_row_hash"),
    )
)

regionsum = (
    bronze.filter((F.col("package_name") == "DISPATCH") & (F.col("table_name") == "REGIONSUM"))
    .select(
        F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias("settlement_datetime"),
        F.upper(F.col("regionid")).alias("region"),
        F.col("intervention").cast("int").alias("intervention"),
        F.col("totaldemand").cast("double").alias("demand_mw"),
        F.col("availablegeneration").cast("double").alias("available_generation_mw"),
        F.col("availableload").cast("double").alias("available_load_mw"),
        F.col("demandforecast").cast("double").alias("demand_forecast_mw"),
        F.col("dispatchablegeneration").cast("double").alias("dispatchable_generation_mw"),
        F.col("dispatchableload").cast("double").alias("dispatchable_load_mw"),
        F.col("netinterchange").cast("double").alias("net_interchange_mw"),
        F.col("excessgeneration").cast("double").alias("excess_generation_mw"),
        F.col("row_hash").alias("regionsum_row_hash"),
    )
)

silver_price_demand = price.join(regionsum, ["settlement_datetime", "region", "intervention"], "left")
silver_price_demand = with_region_name(silver_price_demand)
silver_price_demand = (
    silver_price_demand
    .withColumn("trading_date", F.to_date("settlement_datetime"))
    .withColumn("year", F.year("settlement_datetime"))
    .withColumn("month", F.month("settlement_datetime"))
    .withColumn("day", F.dayofmonth("settlement_datetime"))
    .withColumn("interval_hour", F.hour("settlement_datetime"))
    .withColumn("interval_minute", F.minute("settlement_datetime"))
    .withColumn("silver_loaded_datetime", F.current_timestamp())
    .withColumn("run_id", F.lit(run_id))
    .dropDuplicates(["settlement_datetime", "region", "intervention"])
)

silver_price_demand.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_silver_price_demand_5min")
silver_price_demand.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_silver_regional_dispatch")
display(silver_price_demand.orderBy(F.col("settlement_datetime").desc()).limit(20))

## Build Silver Interconnector Flows

This cell extracts interconnector flow records at interval and interconnector grain when Dispatch interconnector rows are available.

In [ ]:
# Interconnector grain: settlement interval x interconnector x intervention.
interconnector = (
    bronze.filter((F.col("package_name") == "DISPATCH") & (F.col("table_name") == "INTERCONNECTORRES"))
    .select(
        F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias("settlement_datetime"),
        F.col("interconnectorid").alias("interconnector_id"),
        F.col("intervention").cast("int").alias("intervention"),
        F.col("meteredmwflow").cast("double").alias("metered_flow_mw"),
        F.col("mwflow").cast("double").alias("flow_mw"),
        F.col("mwlosses").cast("double").alias("losses_mw"),
        F.col("marginalvalue").cast("double").alias("marginal_value"),
        F.to_date(F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")).alias("trading_date"),
        F.current_timestamp().alias("silver_loaded_datetime"),
        F.lit(run_id).alias("run_id"),
    )
    .dropDuplicates(["settlement_datetime", "interconnector_id", "intervention"])
)

if interconnector.limit(1).count() > 0:
    interconnector.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_silver_interconnector_flows")
    display(interconnector.orderBy(F.col("settlement_datetime").desc()).limit(20))
else:
    print("No interconnector rows available.")

## Build Optional Silver Generation

This cell creates a generation-by-unit Silver table only if suitable DUID and generation columns exist in the Bronze source data.

In [ ]:
# Unit generation is source-dependent. This creates a Silver table only when DUID and generation-like columns are present.
candidate_columns = set(bronze.columns)
if {"duid", "dispatchablegeneration"}.issubset(candidate_columns):
    generation = (
        bronze.filter(F.col("duid") != "")
        .select(
            F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss").alias("settlement_datetime"),
            F.col("duid"),
            F.col("dispatchablegeneration").cast("double").alias("generation_mw"),
            F.to_date(F.to_timestamp("settlementdate", "yyyy/MM/dd HH:mm:ss")).alias("trading_date"),
            F.current_timestamp().alias("silver_loaded_datetime"),
            F.lit(run_id).alias("run_id"),
        )
        .dropDuplicates(["settlement_datetime", "duid"])
    )
    generation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("trading_date").saveAsTable("nem_silver_generation_by_unit")
else:
    print("Generation-by-unit columns not available in current Bronze data.")